# Phase 8 — PySpark Best Practices

**Duration:** Weeks 13–14 | **Goal:** Write production-quality PySpark code that's testable, maintainable, and robust.

---

### What We'll Cover

| Section | Key Skills |
| --- | --- |
| 8.1 Code Organization | Functions, modularity, DRY principle, reusable transforms |
| 8.2 Error Handling | Try/except in PySpark, handling bad data gracefully |
| 8.3 Logging & Monitoring | Print vs logging, tracking row counts, data quality checks |
| 8.4 Idempotent Patterns | MERGE, REPLACE WHERE, safe-to-rerun pipelines |
| 8.5 Testing PySpark Code | Unit testing transforms, test fixtures, assertions |
| 8.6 Configuration Management | Parameterizing notebooks, widgets, environment configs |
| 8.7 Common Anti-Patterns | What NOT to do (and why) |

### Dataset

* `samples.tpch.lineitem` (30M rows) — for real-scale demos
* Custom DataFrames for testing examples

---

**This is the FINAL phase of the PySpark Learning Series.**

After this, you're ready for the **project course** (Medallion Architecture, Pipelines, Jobs, CI/CD).

## 8.1 — Code Organization: Write Like a Software Engineer

### The problem:

Most notebooks are messy:
```python
df = spark.read.table("...")                    # 50 lines of transforms
df2 = df.filter(...).withColumn(...).join(...)   # all in one giant cell
df2.write.mode("overwrite").saveAsTable("...")   # no structure, no reuse
```

### The fix: Extract reusable functions

```python
def clean_names(df):
    """Standardize customer name formatting."""
    return df.withColumn("name", initcap(trim(col("name"))))

def add_revenue(df):
    """Calculate net revenue = price * (1 - discount)."""
    return df.withColumn("revenue", col("l_extendedprice") * (1 - col("l_discount")))

def filter_active(df, cutoff_date):
    """Keep only orders after cutoff date."""
    return df.filter(col("l_shipdate") >= cutoff_date)
```

### Benefits:
* **Testable** — each function can be tested independently
* **Reusable** — same transform in multiple notebooks
* **Readable** — pipeline reads like English: `df.transform(clean_names).transform(add_revenue)`
* **Debuggable** — inspect intermediate results between functions

### The `.transform()` pattern:
```python
# Chain custom functions fluently:
result = (
    spark.read.table("raw_orders")
    .transform(clean_names)
    .transform(add_revenue)
    .transform(filter_active, cutoff_date="2024-01-01")
)
```

In [0]:
# ============================================================
# STEP 1: Code organization — reusable transform functions
# ============================================================
from pyspark.sql.functions import col, initcap, trim, round as spark_round, year, when, lit
from pyspark.sql import DataFrame

SCHEMA = "arnalladatabricks.pyspark_learning"

# --- Define reusable transform functions ---
def add_revenue(df: DataFrame) -> DataFrame:
    """Calculate net revenue = extendedprice * (1 - discount)."""
    return df.withColumn(
        "revenue",
        spark_round(col("l_extendedprice") * (1 - col("l_discount")), 2)
    )

def add_ship_year(df: DataFrame) -> DataFrame:
    """Extract year from ship date."""
    return df.withColumn("ship_year", year(col("l_shipdate")))

def flag_high_value(df: DataFrame, threshold: float = 1000.0) -> DataFrame:
    """Flag orders above threshold as high-value."""
    return df.withColumn(
        "is_high_value",
        when(col("revenue") >= threshold, True).otherwise(False)
    )

def filter_year(df: DataFrame, yr: int) -> DataFrame:
    """Filter to a specific ship year."""
    return df.filter(col("ship_year") == yr)

# --- Use them with .transform() chaining ---
df = spark.read.table("samples.tpch.lineitem").limit(1000)

result = (
    df
    .transform(add_revenue)
    .transform(add_ship_year)
    .transform(flag_high_value, threshold=500.0)
    .transform(filter_year, yr=1995)
)

print("CLEAN PIPELINE using .transform() chaining:")
result.select("l_orderkey", "l_shipdate", "ship_year", "revenue", "is_high_value").show(10)
print(f"Rows after all transforms: {result.count()}")
print("\n→ Each function is testable, reusable, and readable!")
print("  Pipeline reads: add_revenue → add_ship_year → flag_high_value → filter_year")

## 8.2 — Error Handling: Fail Gracefully

### Levels of error handling in PySpark:

| Level | What to handle | Pattern |
| --- | --- | --- |
| **Data-level** | Bad rows, nulls, type mismatches | `PERMISSIVE` mode, `try_*` functions, `coalesce` |
| **Transform-level** | Division by zero, impossible casts | `when/otherwise`, `try_divide`, `try_cast` |
| **Job-level** | Table not found, permission errors | `try/except` around Spark actions |

### Data-level: Handle bad values inline
```python
# Safe division (returns NULL instead of error)
df.withColumn("ratio", try_divide(col("a"), col("b")))

# Safe casting
df.withColumn("amount", expr("try_cast(raw_value AS double)"))

# Default for nulls
df.withColumn("status", coalesce(col("status"), lit("unknown")))
```

### Job-level: Wrap risky operations
```python
try:
    df = spark.read.table("might_not_exist")
except Exception as e:
    print(f"Table not found: {e}")
    df = spark.createDataFrame([], schema)  # empty fallback
```

### Never do:
```python
# ❌ Catching everything silently
try:
    complex_pipeline()
except:
    pass  # Swallows errors — you'll never know what failed!
```

In [0]:
# ============================================================
# STEP 2: Error handling — graceful data and job-level patterns
# ============================================================
from pyspark.sql.functions import col, coalesce, lit, expr, when

# --- Data-level: handle bad values inline ---
print("1. DATA-LEVEL ERROR HANDLING:")

bad_data = [
    (1, "100.50", "10"),
    (2, "abc", "5"),          # can't cast to double
    (3, "200.00", "0"),       # division by zero
    (4, None, "8"),           # null value
    (5, "50.75", None),       # null divisor
]

df_bad = spark.createDataFrame(bad_data, ["id", "amount_str", "qty_str"])

df_safe = df_bad.select(
    "id",
    # Safe casting (NULL instead of error)
    expr("try_cast(amount_str AS double)").alias("amount"),
    expr("try_cast(qty_str AS int)").alias("qty"),
).withColumn(
    # Safe division (NULL instead of divide-by-zero)
    "price_per_unit", expr("try_divide(amount, qty)")
).withColumn(
    # Default for nulls
    "amount_final", coalesce(col("amount"), lit(0.0))
)

print("Raw data:")
df_bad.show()
print("After safe transforms (errors become NULL, not crashes):")
df_safe.show()

# --- Job-level: wrap risky operations ---
print("\n2. JOB-LEVEL ERROR HANDLING:")

def safe_read_table(table_name: str):
    """Try to read a table; return None if it doesn't exist."""
    try:
        df = spark.read.table(table_name)
        print(f"  ✅ Successfully read {table_name} ({df.count():,} rows)")
        return df
    except Exception as e:
        print(f"  ❌ Failed to read {table_name}: {type(e).__name__}")
        return None

# This exists:
df1 = safe_read_table("samples.tpch.nation")
# This doesn't:
df2 = safe_read_table("samples.tpch.nonexistent_table")

print("\n→ Production code always handles missing tables, bad schemas, permissions.")

## 8.3 — Logging & Monitoring: Know What Your Pipeline Did

### Why logging matters:

Without it, when something goes wrong at 3 AM, you have NO idea what happened.

### Three things to always log:

1. **Row counts** — before and after each major transform
2. **Data quality** — null counts, unexpected values, duplicates
3. **Timing** — how long each stage took

### Pattern:
```python
def log_stage(df, stage_name):
    count = df.count()
    print(f"[{stage_name}] Rows: {count:,}")
    return df

result = (
    df
    .transform(clean_data)
    .transform(lambda d: log_stage(d, "After clean"))
    .transform(enrich_data)
    .transform(lambda d: log_stage(d, "After enrich"))
)
```

### Data quality checks:
```python
def check_no_nulls(df, columns, stage=""):
    for c in columns:
        null_count = df.filter(col(c).isNull()).count()
        if null_count > 0:
            print(f"  ⚠️ WARNING [{stage}]: {c} has {null_count:,} nulls")
        else:
            print(f"  ✅ [{stage}]: {c} has no nulls")
```

In [0]:
# ============================================================
# STEP 3: Logging & data quality monitoring
# ============================================================
import time
from pyspark.sql.functions import col, count, sum as spark_sum, when, isnan, isnull

# --- Row count logging between stages ---
def log_stage(df, stage_name):
    """Log row count at a pipeline stage."""
    cnt = df.count()
    print(f"  [{stage_name}] Rows: {cnt:,}")
    return df

# --- Data quality checker ---
def check_quality(df, stage_name=""):
    """Check for nulls in all columns and report."""
    print(f"\n  Quality check [{stage_name}]:")
    total = df.count()
    for c in df.columns[:6]:  # check first 6 columns
        null_count = df.filter(col(c).isNull()).count()
        pct = (null_count / total * 100) if total > 0 else 0
        status = "✅" if null_count == 0 else "⚠️"
        if null_count > 0:
            print(f"    {status} {c}: {null_count:,} nulls ({pct:.1f}%)")
    print(f"    Total rows: {total:,}")
    return df

# --- Timed execution ---
def timed(label, func):
    """Time a function and print duration."""
    start = time.time()
    result = func()
    elapsed = time.time() - start
    print(f"  ⏱ {label}: {elapsed:.2f}s")
    return result

# --- Demo the logging pipeline ---
print("PIPELINE WITH LOGGING:")
print("="*50)

df = spark.read.table("samples.tpch.lineitem")

result = (
    df
    .transform(lambda d: log_stage(d, "Raw input"))
    .filter(col("l_shipdate") >= "1995-01-01")
    .transform(lambda d: log_stage(d, "After date filter"))
    .filter(col("l_returnflag") == "N")
    .transform(lambda d: log_stage(d, "After returnflag filter"))
    .transform(add_revenue)
    .transform(lambda d: log_stage(d, "After add_revenue"))
)

# Quality check on final result
check_quality(result.limit(10000), "Final output sample")

print("\n→ In production, these logs go to a monitoring system (not just print).")
print("  Use Python's `logging` module + Databricks log delivery for real pipelines.")

## 8.4 — Idempotent Patterns: Safe to Re-Run

### What is idempotency?

Running the same code twice produces the SAME result. No duplicates, no corruption.

### ❌ Non-idempotent (dangerous):
```python
# Append mode — running twice = double the data!
df.write.mode("append").saveAsTable("target")
```

### ✅ Idempotent patterns:

| Pattern | When to use | How |
| --- | --- | --- |
| **Overwrite** | Full refresh (small tables) | `.mode("overwrite").saveAsTable(...)` |
| **MERGE** | Incremental upserts | `MERGE INTO target USING source ON key WHEN MATCHED UPDATE WHEN NOT MATCHED INSERT` |
| **REPLACE WHERE** | Overwrite a partition/window | `.option("replaceWhere", "date = '2024-01-01'").mode("overwrite")` |
| **CREATE OR REPLACE** | Table structure changes | `CREATE OR REPLACE TABLE ...` |

### The golden rule:

> If your pipeline crashes halfway and you re-run it, the final state should be **identical** to a successful single run.

### MERGE = the production workhorse:
```python
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, "target")
dt.alias("t").merge(
    source_df.alias("s"),
    "t.id = s.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()
# Run this 100 times → same result every time!
```

In [0]:
# ============================================================
# STEP 4: Idempotent patterns — safe to re-run
# ============================================================
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp, lit

SCHEMA = "arnalladatabricks.pyspark_learning"

# --- Setup: create a target table ---
print("SETUP: Creating target table...")
initial_data = [
    (1, "Alice", 100.0, "2024-01-01"),
    (2, "Bob", 200.0, "2024-01-01"),
    (3, "Charlie", 150.0, "2024-01-01"),
]
df_target = spark.createDataFrame(initial_data, ["id", "name", "amount", "date"])
df_target.write.mode("overwrite").saveAsTable(f"{SCHEMA}.idempotent_demo")
print(f"  Target: {spark.read.table(f'{SCHEMA}.idempotent_demo').count()} rows")

# --- Incoming source data (mix of updates + new rows) ---
source_data = [
    (2, "Bob", 250.0, "2024-01-02"),    # UPDATE: Bob's amount changed
    (3, "Charlie", 150.0, "2024-01-01"),# NO CHANGE
    (4, "Diana", 300.0, "2024-01-02"),  # NEW: insert
    (5, "Eve", 175.0, "2024-01-02"),    # NEW: insert
]
df_source = spark.createDataFrame(source_data, ["id", "name", "amount", "date"])

# --- MERGE: idempotent upsert ---
print("\n1. MERGE (idempotent upsert):")
dt = DeltaTable.forName(spark, f"{SCHEMA}.idempotent_demo")
dt.alias("t").merge(
    df_source.alias("s"),
    "t.id = s.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

result1 = spark.read.table(f"{SCHEMA}.idempotent_demo")
print(f"  After MERGE: {result1.count()} rows")
result1.orderBy("id").show()

# --- Run MERGE AGAIN (proves idempotency) ---
print("2. Run MERGE AGAIN (same source data):")
dt.alias("t").merge(
    df_source.alias("s"),
    "t.id = s.id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

result2 = spark.read.table(f"{SCHEMA}.idempotent_demo")
print(f"  After 2nd MERGE: {result2.count()} rows (same! no duplicates)")

# --- REPLACE WHERE: overwrite only a specific partition ---
print("\n3. REPLACE WHERE (overwrite one date only):")
new_jan02 = spark.createDataFrame([
    (2, "Bob", 275.0, "2024-01-02"),
    (4, "Diana", 310.0, "2024-01-02"),
    (5, "Eve", 180.0, "2024-01-02"),
], ["id", "name", "amount", "date"])

new_jan02.write \
    .mode("overwrite") \
    .option("replaceWhere", "date = '2024-01-02'") \
    .saveAsTable(f"{SCHEMA}.idempotent_demo")

result3 = spark.read.table(f"{SCHEMA}.idempotent_demo")
print(f"  After REPLACE WHERE: {result3.count()} rows")
result3.orderBy("id").show()
print("→ Only 2024-01-02 rows were replaced. 2024-01-01 rows untouched!")
print("  Run again = same result. That's idempotency.")

## 8.5 — Testing PySpark Code

### Why test?

You changed one line → broke a downstream dashboard → VP gets wrong numbers → bad day.

### What to test:

| Test type | What it checks | Example |
| --- | --- | --- |
| **Unit test** | Single transform function | `add_revenue()` produces correct column |
| **Schema test** | Output has expected columns/types | Result has `revenue` as DoubleType |
| **Data quality test** | No nulls in key columns, valid ranges | `revenue >= 0` for all rows |
| **Integration test** | Full pipeline produces expected output | End-to-end with test data |

### Testing pattern in notebooks:
```python
def test_add_revenue():
    # Arrange: create test input
    test_df = spark.createDataFrame([
        (100.0, 0.1),   # expected: 100 * (1-0.1) = 90.0
        (200.0, 0.0),   # expected: 200 * (1-0.0) = 200.0
    ], ["l_extendedprice", "l_discount"])
    
    # Act: run the function
    result = test_df.transform(add_revenue)
    
    # Assert: check output
    rows = result.collect()
    assert rows[0]["revenue"] == 90.0, f"Expected 90.0, got {rows[0]['revenue']}"
    assert rows[1]["revenue"] == 200.0, f"Expected 200.0, got {rows[1]['revenue']}"
    print("✅ test_add_revenue passed!")

test_add_revenue()
```

In [0]:
# ============================================================
# STEP 5: Testing PySpark transforms
# ============================================================

# --- Unit tests for our transform functions ---
def test_add_revenue():
    """Test that revenue = extendedprice * (1 - discount)."""
    test_df = spark.createDataFrame([
        (100.0, 0.1),   # 100 * 0.9 = 90.0
        (200.0, 0.0),   # 200 * 1.0 = 200.0
        (50.0, 0.25),   # 50 * 0.75 = 37.5
    ], ["l_extendedprice", "l_discount"])
    
    result = test_df.transform(add_revenue)
    rows = result.select("revenue").collect()
    
    assert rows[0][0] == 90.0, f"Expected 90.0, got {rows[0][0]}"
    assert rows[1][0] == 200.0, f"Expected 200.0, got {rows[1][0]}"
    assert rows[2][0] == 37.5, f"Expected 37.5, got {rows[2][0]}"
    print("✅ test_add_revenue passed!")

def test_flag_high_value():
    """Test high value flag with custom threshold."""
    test_df = spark.createDataFrame([
        (500.0,), (999.0,), (1000.0,), (1500.0,)
    ], ["revenue"])
    
    result = test_df.transform(flag_high_value, threshold=1000.0)
    flags = [row["is_high_value"] for row in result.collect()]
    
    assert flags == [False, False, True, True], f"Got {flags}"
    print("✅ test_flag_high_value passed!")

def test_filter_year():
    """Test year filtering."""
    test_df = spark.createDataFrame([
        (1994,), (1995,), (1995,), (1996,)
    ], ["ship_year"])
    
    result = test_df.transform(filter_year, yr=1995)
    assert result.count() == 2, f"Expected 2 rows, got {result.count()}"
    print("✅ test_filter_year passed!")

def test_schema_output():
    """Test that pipeline produces expected schema."""
    test_df = spark.createDataFrame([
        (100.0, 0.1, "1995-06-15"),
    ], ["l_extendedprice", "l_discount", "l_shipdate"])
    
    result = test_df.transform(add_revenue).transform(add_ship_year)
    
    assert "revenue" in result.columns, "Missing 'revenue' column"
    assert "ship_year" in result.columns, "Missing 'ship_year' column"
    assert result.schema["revenue"].dataType.simpleString() == "double"
    print("✅ test_schema_output passed!")

# --- Run all tests ---
print("RUNNING UNIT TESTS:")
print("="*50)
test_add_revenue()
test_flag_high_value()
test_filter_year()
test_schema_output()
print("\n🎉 All tests passed!")
print("\n→ In production, use pytest + databricks-connect for CI/CD.")
print("  These tests run in your pipeline BEFORE writing to production tables.")

## 8.6 — Common Anti-Patterns: What NOT to Do

### ❌ Anti-patterns and their fixes:

| Anti-Pattern | Why it's bad | Fix |
| --- | --- | --- |
| `df.collect()` in a loop | Pulls ALL data to driver → OOM | Use DataFrame ops (stay distributed) |
| `.count()` after every transform | Triggers a full job each time | Count only at checkpoints |
| Calling UDFs for simple ops | Python UDF = slow serialization | Use built-in functions |
| `import *` from pyspark.sql.functions | Namespace pollution, hard to debug | Import only what you need |
| `.cache()` everything | Wastes memory, GC pressure | Cache only if reused 2+ times |
| Giant cells (100+ lines) | Untestable, unreadable | Split into functions |
| Hardcoded paths/values | Breaks across environments | Use parameters/widgets |
| No schema on read | `inferSchema` is slow + unreliable | Always provide explicit schema |
| `append` without deduplication | Re-runs create duplicates | Use MERGE or REPLACE WHERE |
| Filtering AFTER joins | Shuffles unnecessary data | Filter early (before joins) |

### The `.collect()` trap:
```python
# ❌ TERRIBLE: pulls millions of rows to driver
for row in df.collect():
    process(row)

# ✅ GOOD: keep everything distributed
df.withColumn("processed", some_function(col("value")))
```

### The "count() everywhere" trap:
```python
# ❌ Slow: 4 full scans
print(df.count())
df2 = df.filter(...)
print(df2.count())
df3 = df2.join(...)
print(df3.count())

# ✅ Better: log counts only at critical stages
df3 = df.filter(...).join(...)
print(f"Final: {df3.count():,}")  # one scan at the end
```

In [0]:
# ============================================================
# STEP 6: Anti-patterns — bad vs good patterns side by side
# ============================================================
import time
from pyspark.sql.functions import col, upper, udf, pandas_udf
from pyspark.sql.types import StringType

df = spark.read.table("samples.tpch.lineitem").limit(100_000)

# --- Anti-pattern 1: UDF vs built-in ---
print("ANTI-PATTERN: Python UDF vs Built-in Function")
print("="*50)

# Bad: Python UDF for simple string operation
@udf(returnType=StringType())
def upper_udf(s):
    return s.upper() if s else None

# Good: Built-in function
start = time.time()
df.withColumn("mode_upper", upper(col("l_shipmode"))).count()
builtin_time = time.time() - start

start = time.time()
df.withColumn("mode_upper", upper_udf(col("l_shipmode"))).count()
udf_time = time.time() - start

print(f"  Built-in upper():  {builtin_time:.2f}s")
print(f"  Python UDF:        {udf_time:.2f}s")
print(f"  → Built-in is {udf_time/builtin_time:.1f}x faster!\n")

# --- Anti-pattern 2: Filter early vs late ---
print("ANTI-PATTERN: Filter Early vs Filter Late")
print("="*50)

df_orders = spark.read.table("samples.tpch.orders")

# Bad: join first, filter later
start = time.time()
df_late = df.join(df_orders, df["l_orderkey"] == df_orders["o_orderkey"]) \
    .filter(col("l_shipmode") == "AIR")
df_late.count()
late_time = time.time() - start

# Good: filter first, join less data
start = time.time()
df_early = df.filter(col("l_shipmode") == "AIR") \
    .join(df_orders, df["l_orderkey"] == df_orders["o_orderkey"])
df_early.count()
early_time = time.time() - start

print(f"  Filter AFTER join:  {late_time:.2f}s")
print(f"  Filter BEFORE join: {early_time:.2f}s")
print(f"  → Filtering early reduces shuffle data!\n")

print("KEY TAKEAWAYS:")
print("  1. Use built-in functions (not UDFs) whenever possible")
print("  2. Filter early to reduce data before expensive ops")
print("  3. Don't .collect() large DataFrames")
print("  4. Don't .count() at every step (only at checkpoints)")
print("  5. Always use MERGE/REPLACE WHERE (not bare append)")

## Phase 8 Checkpoint

**Can you do these?**

- [ ] Write reusable transform functions and chain them with `.transform()`
- [ ] Handle bad data gracefully with `try_cast`, `try_divide`, `coalesce`
- [ ] Add logging (row counts, timing, quality checks) to a pipeline
- [ ] Explain why MERGE is idempotent but APPEND is not
- [ ] Write REPLACE WHERE to overwrite only a specific date/partition
- [ ] Write unit tests for transform functions (Arrange → Act → Assert)
- [ ] Identify and fix common anti-patterns (UDF, filter late, collect, cache all)
- [ ] Explain the difference between data-level vs job-level error handling

---

**Key takeaways:**
1. **Functions > giant cells** — extract transforms, chain with `.transform()`
2. **Fail gracefully** — use `try_*` functions, handle nulls, wrap risky reads
3. **Log everything** — row counts, quality checks, timing at key stages
4. **Idempotent always** — MERGE or REPLACE WHERE, never bare append
5. **Test before deploy** — unit tests catch bugs before they hit production
6. **Built-ins > UDFs** — 10-100x faster, use UDFs only as last resort

---

**🎓 Congratulations! You’ve completed the entire PySpark Learning Series (Phases 0–8)!**

